In [0]:
source_path = "abfss://retail-container@retailstorage2003.dfs.core.windows.net/bronze/"

### Read the raw data from bronze layer

In [0]:

df_transactions = spark.read.parquet(f"{source_path}transaction/")
df_stores = spark.read.parquet(f"{source_path}store/")
df_products = spark.read.parquet(f"{source_path}product/")
df_customers = spark.read.parquet(f"{source_path}customer/Pankaj-Chadda/Data-Engineer/refs/heads/main/")

In [0]:
df_transactions.show()

+--------------+-----------+----------+--------+--------+----------------+
|transaction_id|customer_id|product_id|store_id|quantity|transaction_date|
+--------------+-----------+----------+--------+--------+----------------+
|             1|        127|         8|       4|       4|      2025-03-31|
|             2|        105|         3|       4|       5|      2024-11-12|
|             3|        116|         2|       2|       3|      2025-05-01|
|             4|        120|         8|       1|       1|      2024-11-02|
|             5|        105|         5|       2|       1|      2025-03-17|
|             6|        110|         7|       3|       5|      2025-01-04|
|             7|        110|         7|       2|       5|      2025-01-01|
|             8|        126|         7|       5|       2|      2025-06-08|
|             9|        123|         1|       3|       2|      2024-10-08|
|            10|        124|         2|       2|       5|      2024-08-27|
|            11|        1

### Creating silver layer - data cleaning

In [0]:
from pyspark.sql.functions import col

# Typecasting
df_transactions = df_transactions.select(
    col("transaction_id").cast("int"),
    col("customer_id").cast("int"),
    col("product_id").cast("int"),
    col("store_id").cast("int"),
    col("quantity").cast("int"),
    col("transaction_date").cast("date"),

)
df_products = df_products.select(
    col("product_id").cast("int"),
    col("product_name"),
    col("category"),
    col("price").cast("double")
)
df_stores = df_stores.select(
    col("store_id").cast("int"),
    col("store_name"),
    col("location")
)
df_customers = df_customers.select(
    "customer_id","first_name","last_name","email","city","registration_date"
).dropDuplicates(["customer_id"])



In [0]:
df_transactions.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_date: date (nullable = true)



### Joining the tables

In [0]:
from pyspark.sql.functions import col

df_silver = df_transactions\
    .join(df_customers,"customer_id")\
    .join(df_products,"product_id")\
    .join(df_stores,"store_id")\
    .withColumn("total_amount",col("quantity") * col("price"))

In [0]:
display(df_silver)

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,first_name,last_name,email,city,registration_date,product_name,category,price,store_name,location,total_amount
3,1,102,11,2,2024-08-11,Nina,Joshi,user102@example.com,Mumbai,2024-01-21,Wireless Mouse,Electronics,799.99,Tech World Outlet,Bangalore,1599.98
3,3,104,13,4,2025-05-04,Karan,Patel,user104@example.com,Hyderabad,2024-02-05,Yoga Mat,Fitness,499.0,Tech World Outlet,Bangalore,1996.0
5,8,109,17,5,2024-07-10,Pooja,Mehta,user109@example.com,Delhi,2024-04-01,Desk Organizer,Accessories,399.0,Mega Plaza,Chennai,1995.0
5,7,101,28,3,2024-11-15,Ravi,Yadav,user101@example.com,Delhi,2023-09-14,Smartwatch,Electronics,4999.0,Mega Plaza,Chennai,14997.0
1,5,108,12,4,2025-05-26,Rahul,Verma,user108@example.com,Kolkata,2023-08-19,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,596.0
2,7,110,7,5,2025-01-01,Deepak,Nair,user110@example.com,Mumbai,2023-10-14,Smartwatch,Electronics,4999.0,High Street Store,Delhi,24995.0
4,1,120,14,5,2024-07-17,Alka,Mishra,user120@example.com,Hyderabad,2023-12-01,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,3999.95
5,6,121,15,5,2025-05-19,Sanjay,Patel,user121@example.com,Chennai,2024-01-10,Water Bottle,Fitness,299.0,Mega Plaza,Chennai,1495.0
2,6,118,16,4,2024-11-29,Vikram,Mehta,user118@example.com,Mumbai,2023-04-22,Water Bottle,Fitness,299.0,High Street Store,Delhi,1196.0
5,1,125,24,1,2024-07-14,Tarun,Verma,user125@example.com,Delhi,2024-04-05,Wireless Mouse,Electronics,799.99,Mega Plaza,Chennai,799.99


### Dump the data to ADLS location 

In [0]:
silver_path = "abfss://retail-container@retailstorage2003.dfs.core.windows.net/silver/"

df_silver.write.mode("overwrite").format("delta").save(silver_path)

### Create silver dataset

In [0]:
spark.sql("""DROP TABLE IF EXISTS RETAIL_SILVER_CLEANED""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS RETAIL_SILVER_CLEANED
USING DELTA
LOCATION '{silver_path}'
""")

DataFrame[]

In [0]:
spark.sql("SELECT * FROM RETAIL_SILVER_CLEANED").show()

+--------+----------+-----------+--------------+--------+----------------+----------+---------+-------------------+---------+-----------------+-----------------+-----------+-------+-------------------+---------+------------------+
|store_id|product_id|customer_id|transaction_id|quantity|transaction_date|first_name|last_name|              email|     city|registration_date|     product_name|   category|  price|         store_name| location|      total_amount|
+--------+----------+-----------+--------------+--------+----------------+----------+---------+-------------------+---------+-----------------+-----------------+-----------+-------+-------------------+---------+------------------+
|       3|         1|        102|            11|       2|      2024-08-11|      Nina|    Joshi|user102@example.com|   Mumbai|       2024-01-21|   Wireless Mouse|Electronics| 799.99|  Tech World Outlet|Bangalore|           1599.98|
|       3|         3|        104|            13|       4|      2025-05-04|  

In [0]:
df_silver.show()

+--------+----------+-----------+--------------+--------+----------------+----------+---------+-------------------+----------+---------+-----------------+-----------------+-----------+-------+-------------------+---------+------------+
|store_id|product_id|customer_id|transaction_id|quantity|transaction_date|first_name|last_name|              email|     phone|     city|registration_date|     product_name|   category|  price|         store_name| location|total_amount|
+--------+----------+-----------+--------------+--------+----------------+----------+---------+-------------------+----------+---------+-----------------+-----------------+-----------+-------+-------------------+---------+------------+
|       5|         7|        101|            28|       3|      2024-11-15|      Ravi|    Yadav|user101@example.com|9887654321|    Delhi|       2023-09-14|       Smartwatch|Electronics|4999.00|         Mega Plaza|  Chennai|    14997.00|
|       3|         1|        102|            11|       2

### Gold Layer

In [0]:
# Load clean transaction from silver layer
silver_df = spark.read.format("delta").load(silver_path)

In [0]:
from pyspark.sql.functions import sum, avg, countDistinct

gold_df = silver_df.groupBy(
    "transaction_date",
    "product_id","product_name","category",
    "store_id","store_name","location"
).agg(
    sum("quantity").alias("total_quantity_sold"),
    sum("total_amount").alias("total_sales_amount"),
    countDistinct("transaction_id").alias("number_of_transactions"),
    avg("total_amount").alias("average_transaction_value")
)
display(gold_df)



transaction_date,product_id,product_name,category,store_id,store_name,location,total_quantity_sold,total_sales_amount,number_of_transactions,average_transaction_value
2024-08-11,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
2025-05-04,3,Yoga Mat,Fitness,3,Tech World Outlet,Bangalore,4,1996.0,1,1996.0
2024-07-10,8,Desk Organizer,Accessories,5,Mega Plaza,Chennai,5,1995.0,1,1995.0
2024-11-15,7,Smartwatch,Electronics,5,Mega Plaza,Chennai,3,14997.0,1,14997.0
2025-05-26,5,Notebook Set,Stationery,1,City Mall Store,Mumbai,4,596.0,1,596.0
2025-01-01,7,Smartwatch,Electronics,2,High Street Store,Delhi,5,24995.0,1,24995.0
2024-07-17,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,5,3999.95,1,3999.95
2025-05-19,6,Water Bottle,Fitness,5,Mega Plaza,Chennai,5,1495.0,1,1495.0
2024-11-29,6,Water Bottle,Fitness,2,High Street Store,Delhi,4,1196.0,1,1196.0
2024-07-14,1,Wireless Mouse,Electronics,5,Mega Plaza,Chennai,1,799.99,1,799.99


In [0]:
gold_path = "abfss://retail-container@retailstorage2003.dfs.core.windows.net/gold/"

gold_df.write.mode("overwrite").format("delta").save(gold_path)


In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS retail_gold_sales_summary
    USING DELTA
    LOCATION '{gold_path}';
    """
)

DataFrame[]

In [0]:
%sql SELECT * FROM retail_gold_sales_summary;

transaction_date,product_id,product_name,category,store_id,store_name,location,total_quantity_sold,total_sales_amount,number_of_transactions,average_transaction_value
2025-05-04,3,Yoga Mat,Fitness,3,Tech World Outlet,Bangalore,4,1996.0,1,1996.0
2024-12-13,8,Desk Organizer,Accessories,4,Downtown Mini Store,Pune,5,1995.0,1,1995.0
2024-08-11,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
2024-11-02,8,Desk Organizer,Accessories,1,City Mall Store,Mumbai,1,399.0,1,399.0
2024-09-05,1,Wireless Mouse,Electronics,4,Downtown Mini Store,Pune,3,2399.9700000000003,1,2399.9700000000003
2024-11-29,6,Water Bottle,Fitness,2,High Street Store,Delhi,4,1196.0,1,1196.0
2024-08-27,2,Bluetooth Speaker,Electronics,2,High Street Store,Delhi,5,6497.45,1,6497.45
2024-11-12,3,Yoga Mat,Fitness,4,Downtown Mini Store,Pune,5,2495.0,1,2495.0
2025-04-30,9,Dumbbell Set,Fitness,4,Downtown Mini Store,Pune,2,3998.0,1,3998.0
2024-10-08,1,Wireless Mouse,Electronics,3,Tech World Outlet,Bangalore,2,1599.98,1,1599.98
